In [ ]:
# Import packages, load the study configuration and the CHIP tables written by
# 11_chip_tables
from pathlib import Path

from SSc_Somatic import configure, load_config, load_study
from SSc_Somatic.chip import analysis as chip_funcs
from SSc_Somatic.chip import plots as chip_plots_funcs

# Study configuration. Point this at another configuration file to analyze a different cohort.
config = configure(load_config(Path('..') / 'config' / 'example_SSc.yaml'))
data = load_study(config, 'chip')

# Cohort labels and column names used below
status_col = config.require('cohort', 'status_column')
meta_id_col = config.require('cohort', 'metadata_sample_column')
age_col = config.require('cohort', 'clinical_columns', 'age')
case_label = config.require('cohort', 'case_label')
control_label = config.require('cohort', 'control_label')
max_age = config.require('chip', 'max_age')
bins, labels = config.age_bins, config.age_labels

# Tables written by 11_chip_tables and loaded for the CHIP analysis
meta_df = data.metadata
snv_df = data.snv_table
processed_df = data.processed_metadata
wl_final_anno_df = data.whitelist_calls
wl_final_merged_df = data.whitelist_merged

In [ ]:
# Preview the annotated CHIP whitelist table and keep only cases and controls
print("First rows of the annotated CHIP whitelist table:")
print(wl_final_anno_df.head())
wl_final_anno_df = wl_final_anno_df.loc[wl_final_anno_df[status_col].isin([case_label, control_label])]
print("Number of participants with CHIP calls after keeping cases and controls:",
      len(wl_final_anno_df[meta_id_col].unique()))

In [ ]:
# Preview the merged table and keep only cases and controls
print("First rows of the merged table (one row per participant):")
print(wl_final_merged_df.head())
wl_final_merged_df = wl_final_merged_df.loc[wl_final_merged_df[status_col].isin([case_label, control_label])]
print("Number of participants in the merged table after keeping cases and controls:",
      len(wl_final_merged_df[meta_id_col].unique()))

In [ ]:
# Restrict the metadata to processed samples and annotate the SNV count table with
# metadata and age groups
print("Annotating the SNV count table with metadata and age groups:")
processed_ids = processed_df['sample_id'].tolist()
meta_df = meta_df.loc[meta_df[meta_id_col].isin(processed_ids)]
snv_anno_ssc_df = chip_funcs.create_snv_anno_ssc_dataframe(snv_df, processed_df, bins, labels)

In [ ]:
# Create age-filtered version (too few samples at or above max_age)
wl_final_merged_df_age = wl_final_merged_df.loc[wl_final_merged_df[age_col] < max_age].copy()
print(f"Merged table restricted to {age_col} below {max_age}: {len(wl_final_merged_df_age)} rows")
print("Number of participants by case or control status:")
print(wl_final_merged_df_age[status_col].value_counts())
print("Number of participants with (1) and without (0) a CHIP whitelist variant:")
print(wl_final_merged_df_age['is_wl'].value_counts())

In [ ]:
# Summarize disease duration by CHIP status and case subtype
print("Disease duration by CHIP status and case subtype:")
duration_results = chip_funcs.summarize_disease_duration_by_chip_and_subtype(wl_final_merged_df_age)

General plots to check VAF distributions and gene-specific CHIP calls.

In [ ]:
# Plot the VAF distribution of CHIP calls and the number of samples with a CHIP variant
# in each gene
print("Figures: VAF distribution of CHIP whitelist variants, and number of samples with a "
      "CHIP variant in each gene (all samples)")
stats = chip_plots_funcs.plot_vaf_and_gene_distribution(wl_final_anno_df, title_suffix="in all samples")
print(f"\nMost common gene: {stats['gene_counts'].index[0]}")
print(f"Number of unique genes: {stats['unique_genes']}")

In [ ]:
# Count controls in each age group and plot CHIP variants by gene and disease group
print("Number of controls in each age group:")
age_bins_normal = chip_funcs.calculate_control_age_distribution(meta_df, bins, labels)

print("Figure: number and proportion of samples with CHIP variants in each gene, by disease group")
gene_subtype_results = chip_plots_funcs.plot_chip_variants_by_subtype(
    wl_final_anno_df,
    snv_anno_ssc_df,
    age_bins_normal,
    labels,
    age_bins=bins
)

In [ ]:
# Plot CHIP occurrence against age in each disease group
print("Figure: CHIP occurrence by age in each disease group, with logistic regression fits "
      "and 95% confidence intervals")
fig, r2_results = chip_plots_funcs.create_chip_occurrence_plot(
    wl_final_merged_df_age,
    output_filename=None
)

In [ ]:
# Compare disease duration between participants with and without CHIP
print("Figure: disease duration by CHIP status, with pairwise Mann-Whitney U tests "
      "(Benjamini-Hochberg FDR correction)")
duration_boxplot_results = chip_plots_funcs.analyze_wl_disease_duration_boxplots(
    wl_final_merged_df_age,
    output_filename=None,
    figsize=(8, 8)
)

In [ ]:
# Odds ratios for CHIP, adjusted for age, in four group comparisons
print("Odds ratios for CHIP, adjusted for age, in four group comparisons, followed by a "
      "forest plot of the results:")
combined_results = chip_funcs.run_combined_chip_analysis(
    wl_final_merged_df_age,
    chip_funcs.calculate_adjusted_odds_ratio,
    output_filename=None
)